# 09 — Sensitivity: regional vs voxelwise lesion features

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                    if (p / "analysis" / "lib").exists())
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import nibabel as nib
from nibabel.processing import resample_from_to
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed
from threadpoolctl import threadpool_limits

from analysis.lib.data import (load_label, restricted_pool,
                               build_demographics_matrix, identify_binary_targets)
from analysis.lib.pipelines import process_fold
from analysis.lib.cv import pooled_cv
from analysis.lib.stats import nadeau_bengio_test, bh_fdr
from analysis.lib.plotting import VARIABLE_LABELS, PALETTE

DATA_DIR   = PROJECT_ROOT / "data"
LACUNA     = DATA_DIR / "lacuna" / "METAVCI_PSCI_LACUNA"
LACUNA_DIR = LACUNA / "derivatives"
OUT_DIR    = PROJECT_ROOT / "outputs" / "predictive_modeling_sensitivity" / "regional_vs_voxelwise"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CLIN_PATH  = DATA_DIR / "tabular" / "clinical_data.csv"

VAR_KEPT = 0.95
N_JOBS   = 8          # parallel folds; each limited to 2 BLAS threads
# representation -> (voxelwise file suffix, parcel loader args)
REPS = {
    "Lesion": ("space-MNI152NLin6Asym_mask.nii.gz",                              ("rd",  "damagepct")),
    "fLNM":   ("method-fnm_space-MNI152NLin6Asym_desc-tmap.nii.gz",              ("fnm", "tmap")),
    "sLNM":   ("method-snm_space-MNI152NLin6Asym_desc-disconnectionpct.nii.gz", ("snm", "disconnectionpct")),
}

In [ ]:
# Gram matrices K = X X^T of the voxelwise maps (imaging only; cached).
# Voxels that are zero in every patient do not contribute to K, so the full
# 2 mm grid is used without any data-derived mask. Lesion masks (1 mm) are
# resampled to the 2 mm grid (fraction of each 2 mm voxel infarcted).
SUBJECTS = sorted(p.name[4:] for p in LACUNA.glob("sub-*")
                  if (p / "ses-01" / "anat" / f"{p.name}_ses-01_label-acuteinfarct_{REPS['Lesion'][0]}").exists())
_ref = nib.load(str(LACUNA / f"sub-{SUBJECTS[0]}" / "ses-01" / "anat"
                    / f"sub-{SUBJECTS[0]}_ses-01_label-acuteinfarct_{REPS['fLNM'][0]}"))
GRID = (_ref.shape, _ref.affine)


def load_map(sid, rep):
    f = LACUNA / f"sub-{sid}" / "ses-01" / "anat" / f"sub-{sid}_ses-01_label-acuteinfarct_{REPS[rep][0]}"
    img = nib.load(str(f))
    if rep == "Lesion":
        img = resample_from_to(nib.Nifti1Image(np.asarray(img.dataobj, dtype=np.float32), img.affine),
                               GRID, order=1)
    return np.nan_to_num(np.asarray(img.dataobj, dtype=np.float32)).ravel()


def gram(rep, chunk=200):
    out = OUT_DIR / f"gram_{rep}.npz"
    if out.exists():
        z = np.load(out, allow_pickle=True)
        return pd.DataFrame(z["K"], index=z["subjects"], columns=z["subjects"])
    X = np.empty((len(SUBJECTS), int(np.prod(GRID[0]))), dtype=np.float32)
    for i in range(0, len(SUBJECTS), chunk):
        X[i:i + chunk] = np.vstack(Parallel(n_jobs=-1)(
            delayed(load_map)(s, rep) for s in SUBJECTS[i:i + chunk]))
    X = X[:, np.abs(X).sum(axis=0) > 0]
    K = np.zeros((len(SUBJECTS), len(SUBJECTS)))
    for j in range(0, X.shape[1], 20000):
        B = X[:, j:j + 20000].astype(np.float64)
        K += B @ B.T
    np.savez(out, K=K, subjects=np.array(SUBJECTS), n_voxels=X.shape[1])
    print(f"{rep}: {len(SUBJECTS)} subjects x {X.shape[1]} non-zero voxels")
    return pd.DataFrame(K, index=SUBJECTS, columns=SUBJECTS)


GRAMS = {rep: gram(rep) for rep in REPS}

In [ ]:
def fold_pca(K, tr, te, var_kept=VAR_KEPT):
    """Exact PCA of the training maps from the Gram matrix; test maps projected
    with the training mean and components only."""
    Ktr = K[np.ix_(tr, tr)]
    m = Ktr.mean(axis=0)
    w, U = np.linalg.eigh(Ktr - m[None, :] - m[:, None] + m.mean())
    w, U = w[::-1], U[:, ::-1]
    w = np.clip(w, 0, None)
    k = int(np.searchsorted(np.cumsum(w) / w.sum(), var_kept) + 1)
    Z_tr = U[:, :k] * np.sqrt(w[:k])
    Kte = K[np.ix_(te, tr)]
    Z_te = (Kte - Kte.mean(axis=1, keepdims=True) - m[None, :] + m.mean()) @ U[:, :k] / np.sqrt(w[:k])
    return Z_tr, Z_te, k


def voxel_fold(i, tr, te, K, X_base, y, target, rep):
    with threadpool_limits(2):
        Z_tr, Z_te, k = fold_pca(K, tr, te)
        X = np.vstack([np.hstack([X_base[tr], Z_tr]), np.hstack([X_base[te], Z_te])])
        y_fold = np.concatenate([y[tr], y[te]])   # rows of X are reordered: train, then test
        r = process_fold(i, np.arange(len(tr)), np.arange(len(tr), len(tr) + len(te)),
                         X, y_fold, rep, target, X_base.shape[1])
    r.update(Representation=rep, Format="Voxelwise + PCA", n_components=k)
    return r

In [ ]:
assert CLIN_PATH.exists(), f"missing {CLIN_PATH} — see data/README.md"
clinical_df = pd.read_csv(CLIN_PATH, index_col=0)
acute_pool = restricted_pool(LACUNA_DIR, clinical_df, all_of=["acuteinfarct"])
clinical = clinical_df.loc[acute_pool]
X_demographics, _, _, _, _ = build_demographics_matrix(clinical, exclude_cohort=False)
targets = identify_binary_targets(clinical)
parcel = {rep: load_label(LACUNA_DIR, "acuteinfarct", *REPS[rep][1]).reindex(acute_pool) for rep in REPS}
assert set(map(str, acute_pool)) <= set(GRAMS["Lesion"].index)

scores_csv = OUT_DIR / "scores.csv"
if scores_csv.exists():
    scores = pd.read_csv(scores_csv)
else:
    cv = pooled_cv(seed=42)
    rows = []
    for target in targets:
        valid = clinical[target].notna().values
        y = clinical[target].values[valid].astype(int)
        X_base = X_demographics[valid]
        ids = acute_pool[valid].astype(str)
        splits = list(cv.split(X_base, y))
        for rep in REPS:
            Xp = np.hstack([X_base, parcel[rep].loc[acute_pool[valid]].values])
            for r in Parallel(n_jobs=-1)(delayed(process_fold)(i, tr, te, Xp, y, rep, target, X_base.shape[1])
                                         for i, (tr, te) in enumerate(splits)):
                r.update(Representation=rep, Format="Parcels", n_components=np.nan)
                rows.append(r)
            K = GRAMS[rep].loc[ids, ids].values
            rows.extend(Parallel(n_jobs=N_JOBS)(delayed(voxel_fold)(i, tr, te, K, X_base, y, target, rep)
                                                for i, (tr, te) in enumerate(splits)))
            print(f"{target} | {rep} done")
    scores = pd.DataFrame(rows)
    scores.to_csv(scores_csv, index=False)

summ = []
for (t, rep), g in scores.groupby(["Target", "Representation"]):
    for metric in ("AUPRC", "AUROC"):
        pv = g.pivot_table(index="Fold", columns="Format", values=metric, dropna=False)
        d = pv["Voxelwise + PCA"] - pv["Parcels"]
        nb = nadeau_bengio_test(pv["Voxelwise + PCA"].values, pv["Parcels"].values,
                                n_splits=10, test_size=0.1, alternative="two-sided")
        summ.append({"Target": t, "Representation": rep, "metric": metric,
                     "parcels": pv["Parcels"].mean(), "voxel_pca": pv["Voxelwise + PCA"].mean(),
                     "delta": d.mean(), "se": d.std(ddof=1) / np.sqrt(d.count()), "p_NB": nb["p"],
                     "n_components_median": g.loc[g.Format == "Voxelwise + PCA", "n_components"].median()})
summ = pd.DataFrame(summ)
summ["p_FDR"] = summ.groupby(["Target", "metric"])["p_NB"].transform(lambda s: bh_fdr(s.values))
summ.to_csv(OUT_DIR / "summary.csv", index=False)
print(summ.round(4).to_string(index=False))

In [ ]:
# Each feature-format variant vs the baseline model (notebook 01, identical
# folds): one-sided Nadeau-Bengio, BH-FDR within target and metric
base_scores = pd.read_csv(PROJECT_ROOT / "outputs" / "predictive_modeling_core" / "pooled" / "all_scores.csv")
base_scores = base_scores[base_scores["Model"] == "Baseline"][["Target", "Fold", "AUPRC", "AUROC"]]
rows = []
for (t, rep, fmt), g in scores.groupby(["Target", "Representation", "Format"]):
    m = g.merge(base_scores[base_scores.Target == t], on="Fold", suffixes=("", "_base"))
    for metric in ("AUPRC", "AUROC"):
        nb = nadeau_bengio_test(m[metric].values, m[f"{metric}_base"].values,
                                n_splits=10, test_size=0.1, alternative="greater")
        rows.append({"Target": t, "Representation": rep, "Format": fmt, "metric": metric,
                     "mean_diff": nb["mean_diff"], "p": nb["p"], "n_paired": nb["n_paired"]})
vs_base = pd.DataFrame(rows)
vs_base["p_adj_BH"] = vs_base.groupby(["Target", "metric"])["p"].transform(lambda s: bh_fdr(s.values))
vs_base.to_csv(OUT_DIR / "vs_baseline.csv", index=False)
print(vs_base.assign(sig=vs_base.p_adj_BH < 0.05)
      .pivot_table(index=["Representation", "Format"], columns="metric", values="sig", aggfunc="sum"))

In [ ]:
# Figure: fold-level performance per representation and feature format
sns.set_theme(style="whitegrid", font_scale=1.25,
              rc={"font.weight": "bold", "axes.labelweight": "bold"})
from matplotlib.patches import Patch
REP_LABEL = {"Lesion": "Infarct location", "fLNM": "Functional LNM", "sLNM": "Structural LNM"}
tg = [t for t in VARIABLE_LABELS if t in scores["Target"].unique()]
FORMATS = ["Parcels", "Voxelwise + PCA"]
order = [("Baseline", "Baseline")] + [(f, r) for f in FORMATS for r in REPS]   # baseline, parcels, voxelwise
offs = np.linspace(-0.40, 0.40, len(order) + 1)
offs = {k: (offs[i] + offs[i + 1]) / 2 + {"Baseline": -0.03, "Parcels": 0.0, "Voxelwise + PCA": 0.03}[k[0]]
        for i, k in enumerate(order)}
width = 0.10
BASE = (pd.read_csv(PROJECT_ROOT / "outputs" / "predictive_modeling_core" / "pooled" / "all_scores.csv")
        .query("Model == 'Baseline'"))
fig, axes = plt.subplots(2, 1, figsize=(20, 11), sharex=True, gridspec_kw={"hspace": 0.12})
for ax, metric in zip(axes, ("AUPRC", "AUROC")):
    for x, t in enumerate(tg):
        for (fmt, rep), o in offs.items():
            if fmt == "Baseline":
                v = BASE.loc[BASE.Target == t, metric].dropna().values
            else:
                v = scores.loc[(scores.Target == t) & (scores.Representation == rep)
                               & (scores.Format == fmt), metric].dropna().values
            parts = ax.violinplot(v, positions=[x + o], widths=width, showextrema=False)
            for b in parts["bodies"]:
                b.set_facecolor(PALETTE[rep]); b.set_edgecolor("black"); b.set_linewidth(0.6)
                b.set_alpha(0.35 if fmt == "Voxelwise + PCA" else 0.85)
                if fmt == "Voxelwise + PCA":
                    b.set_hatch("///")
            ax.scatter(x + o, v.mean(), marker="D", s=26, color="white", edgecolor="black", lw=0.9, zorder=3)
            if fmt != "Baseline":
                r_ = vs_base[(vs_base.Target == t) & (vs_base.Representation == rep)
                             & (vs_base.Format == fmt) & (vs_base.metric == metric)].iloc[0]
                if r_.p_adj_BH < 0.05 and r_.mean_diff > 0:
                    ax.scatter(x + o, v.max() + 0.02 * (1 if metric == "AUPRC" else 0.7), marker="p", s=70,
                               color=PALETTE[rep], edgecolor="black", lw=0.8, zorder=4)
    for x in range(1, len(tg)):
        ax.axvline(x - 0.5, color="gray", lw=0.6, alpha=0.6)
    ax.set_ylabel(metric)
    ax.grid(axis="x", visible=False)
fig.supylabel("Predictive performance", fontsize=15, fontweight="bold", x=0.075)
axes[1].set_xticks(range(len(tg))); axes[1].set_xticklabels([VARIABLE_LABELS[t] for t in tg], fontsize=13)
handles = [Patch(facecolor=PALETTE["Baseline"], edgecolor="black", label="Baseline")] + [
    Patch(facecolor=PALETTE[r], edgecolor="black", label=REP_LABEL[r]) for r in REPS] + [
    Patch(facecolor="gray", alpha=0.85, edgecolor="black", label="Baseline + regional features"),
    Patch(facecolor="gray", alpha=0.35, edgecolor="black", hatch="///", label="Baseline + voxelwise maps + PCA"),
    plt.Line2D([0], [0], marker="p", color="w", markerfacecolor="gray", markeredgecolor="black", markersize=10,
               label=r"> Baseline ($\mathbf{p_{FDR}<0.05}$, NB)"),
    plt.Line2D([0], [0], marker="D", color="w", markerfacecolor="white", markeredgecolor="black",
               markersize=8, label="Mean across folds")]
axes[1].legend(handles=handles, bbox_to_anchor=(0.5, -0.13), loc="upper center", ncol=4,
               frameon=True, fontsize=12)
fig.savefig(OUT_DIR / "regional_vs_voxelwise.png", dpi=200, bbox_inches="tight")
fig.savefig(OUT_DIR / "regional_vs_voxelwise.pdf", bbox_inches="tight")
print(f"saved {OUT_DIR / 'regional_vs_voxelwise.png'}")